# One declaration, two estimators

This notebook is the example on the handbook's **Experiments** landing page.

An experiment is a **declaration**, and not a script. You say which parameters are held and which
vary, you give it a stack of methods and a backend, and you get one row for each condition.

The question here is a small one: **what does a resolver do to random traffic, and is the choice
of the resolver important?** Three resolvers (none, MVP, VO) against three fleet sizes (4, 6, 8
aircraft) gives nine conditions.

One declaration covers the nine, but two estimators run them. Traffic with no resolver loses
separation in a large fraction of the runs, and plain Monte Carlo measures that correctly. Traffic
with a resolver loses separation too seldom for Monte Carlo at a usual cost, thus those conditions
use the rare-event estimator. The backend is the only argument that is different between the two
calls below.

In [ ]:
%matplotlib inline
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 130

from opencdarr import (
    M600,
    MVP,
    VO,
    Fixed,
    GnssNavigation,
    Methods,
    ProbabilisticFTR,
    RandomTraffic,
    StateBased,
    Sweep,
    run_experiment,
)
from opencdarr.config import (
    Config,
    ConflictConfig,
    MethodsConfig,
    ScenarioConfig,
    SimulationConfig,
)
from opencdarr.experiment import IPS, MC, Ladder

# The base config gives each parameter that no axis declares: the parameters of the encounter
# distribution, the protected zone and the horizon, and the numerics. A parameter that the
# declaration gives as Fixed or Sweep replaces the value here.
CFG = Config(
    seed=0,
    n_encounters=0,          # the backend sets this
    scenario=ScenarioConfig(aircraft_type="M600", speed=10.0, dcpa_max=50.0, tlos=60.0,
                            pos_ci95=10.0, vel_ci95=1.0),     # a 10 m / 1 m/s GNSS fix
    conflict=ConflictConfig(rpz=50.0, t_lookahead=30.0),
    methods=MethodsConfig(detection="statebased", resolution="mvp", recovery="pastcpa",
                          margin=1.05, bouncing_guard=True),  # only run_one_experiment reads this
    simulation=SimulationConfig(dt=0.5, t_max=600.0, done_timeout=10.0),
)

# The stack that stays the same in each condition. The resolver is not here, because the
# declarations below give it as an axis or as a Fixed.
STACK = Methods(detector=StateBased(), recovery=ProbabilisticFTR(),
                navigation=GnssNavigation(), perf=M600)

# One axis, used by the two calls: the fleet size builds the scenario for that condition.
FLEET = Sweep([4, 6, 8], name="n_aircraft",
              build=lambda n: RandomTraffic(n, r_inner=1000.0, r_outer=1200.0))

## The baseline: no resolver, on Monte Carlo

Without a resolver the aircraft fly their tracks and the losses are frequent, thus plain Monte
Carlo counts them directly. The denominator is the number of runs times the number of aircraft,
and the interval is a Wilson score interval on that fixed denominator.

In [ ]:
t0 = time.time()

baseline = run_experiment(
    {"scenario": FLEET, "resolver": Fixed(None)},
    methods=STACK, backend=MC(n_encounters=500), base_config=CFG, seed=1, n_jobs=-1,
)

for row in baseline.records():
    print(f"N = {row['n_aircraft']}   P(LoS) = {row['p_los']:.3f}   "
          f"95% CI [{row['p_los_lo']:.3f}, {row['p_los_hi']:.3f}]   "
          f"median closest approach {row['median_min_sep']:6.1f} m")
print(f"\n{(time.time() - t0) / 60:.1f} min")

## The two resolvers, on the rare-event estimator

The same fleet axis, a resolver axis with two levels, and a different backend. Nothing else moves.

`Sweep.build` maps each level onto the value that the run needs, thus the table reads the names
and the run receives the objects.

`Ladder` gives the shells from a pilot Monte-Carlo run of **that** condition. This is necessary
here, because a fleet of 8 aircraft loses separation much more frequently than a fleet of 4. One
fixed ladder cannot be correct for the two.

In [ ]:
RESOLVERS = {"MVP": MVP(1.05), "VO": VO(1.05)}

t0 = time.time()

resolved = run_experiment(
    {"scenario": FLEET,
     "resolver": Sweep(["MVP", "VO"], name="resolver", build=RESOLVERS.__getitem__)},
    methods=STACK,
    backend=IPS(shells=Ladder(pilot=500), n_particles=500, reps=5),
    base_config=CFG, seed=1, n_jobs=-1,
)

for row in resolved.records():
    print(f"N = {row['n_aircraft']}  {row['resolver']:>3}   P(LoS) = {row['p_los']:.2e}   "
          f"95% CI [{row['p_los_lo']:.2e}, {row['p_los_hi']:.2e}]   "
          f"collapsed {row['n_collapsed']}/{row['reps']}")
print(f"\n{(time.time() - t0) / 60:.1f} min")

## The two tables together

The rows come from two estimators, thus the columns are not the same. Monte Carlo reports counts
and a Wilson interval on a denominator that the design fixed. IPS reports a replicated probability
with an interval in log space, and a count of the replications that collapsed. The quantity that
the two estimate is the same, and the
[validation](https://fazlurnu.github.io/opencdarr.github.io/estimators/rare-event/validation/)
page is the evidence for that.

In [ ]:
rows = []
for row in baseline.records():
    rows.append({"n_aircraft": row["n_aircraft"], "resolver": "none", "backend": "MC",
                 "p_los": row["p_los"], "lo": row["p_los_lo"], "hi": row["p_los_hi"]})
for row in resolved.records():
    rows.append({"n_aircraft": row["n_aircraft"], "resolver": row["resolver"], "backend": "IPS",
                 "p_los": row["p_los"], "lo": row["p_los_lo"], "hi": row["p_los_hi"]})

for r in sorted(rows, key=lambda r: (r["n_aircraft"], r["resolver"])):
    print(f"N = {r['n_aircraft']}  {r['resolver']:>4}  {r['backend']:>3}   "
          f"P(LoS) = {r['p_los']:.2e}   [{r['lo']:.2e}, {r['hi']:.2e}]")

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 4.2))
sizes = [4, 6, 8]

for name in ("none", "MVP", "VO"):
    cells = [r for r in rows if r["resolver"] == name]
    cells.sort(key=lambda r: r["n_aircraft"])
    p = np.array([r["p_los"] for r in cells])
    lo = np.array([r["lo"] for r in cells])
    hi = np.array([r["hi"] for r in cells])
    line, = ax.plot(sizes, p, marker="o", label=name)
    ax.fill_between(sizes, lo, hi, alpha=0.2, color=line.get_color())

ax.set_yscale("log")
ax.set_xticks(sizes)
ax.set_xlabel("aircraft in the disc")
ax.set_ylabel("P(LoS) per aircraft")
ax.set_box_aspect(1)
ax.legend()
fig.tight_layout()

# The handbook keeps its figures in the site repository, beside this one.
out = Path("../../../opencdarr.github.io/docs/assets/img/experiment-random-traffic.png")
if out.parent.is_dir():
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print(f"wrote {out}")

## What the same declaration gives you next

Nothing above knows which resolver it received. Thus an algorithm that you wrote this morning is
swept in the same way as `MVP` and `VO` here, against the same encounters and from the same seeds.

To change the estimator, change one argument. To change the traffic, change the scenario axis. The
rest of the declaration does not move.